### Pinecone Vector DB



In [68]:
from google.colab import userdata
import os

In [69]:
## Create your index and apikey from here https://www.pinecone.io/
api_key = userdata.get('pinecone_api_key')

In [70]:
# !pip install -qU langchain langchain-pinecone langchain_google_genai langchain_groq langchain_community langchain

In [71]:
from pinecone import Pinecone
pc = Pinecone(api_key=api_key)

In [72]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)

In [73]:
from pinecone import ServerlessSpec

index_name = "rag-3072"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=3072,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [74]:
index

In [75]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [76]:
vector_store

In [77]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [78]:
vector_store.add_documents(documents=documents)

['6b325b43-9111-45fc-bccb-8df404db33d7',
 'bdb87f9c-9339-4c11-b303-396a7dcffe55',
 '7a7f231b-76a5-46c0-b309-88c669de65ad',
 'f0f8fbab-556e-465b-abad-f2a6f0333b34',
 '863697c2-04dc-4494-b36a-96a295a27fd8',
 '04b9b643-dbdb-4922-8644-3f9c3f198cee',
 '7c9fa8cf-2480-44b6-9e84-0b0bc4c54792',
 'fb400201-26d5-46d3-b2d6-cdfa9bb495cf',
 '46127e23-7153-46de-bd6b-b6a0d45b631a',
 '846f4b29-3a07-4d0e-8a6e-2230ce9277cf']

In [79]:
### Query Directly
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": "tweet"},
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]
* Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]


In [80]:
results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow?", k=1, filter={"source": "news"}
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.721077] The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees. [{'source': 'news'}]


In [81]:
### Retriever
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.4},
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='f0f8fbab-556e-465b-abad-f2a6f0333b34', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]